# Utilidade - Do desenho ao Gcode

In [ ]:
from components import gui
from components import files
from components import path_tools
from components import gcode_tools
from components import images_tools as it
from components import points_tools as pt
from components import morphology_tools as mt
from components import skeleton as skt
from components.timer import Timer
from components.layer import Island, Layer
from components.offset import Offset
from os import getcwd, name
import cv2
import numpy as np

folders = files.System_Paths(getcwd())
path_input, file_name = gui.load_model(folders)

gray = cv2.imread(path_input, 0)
_, binary = cv2.threshold(gray, 244, 255, cv2.THRESH_BINARY_INV)
img = binary.astype(bool)
img = path_tools.one_pixel_wide(img)
img,a,b = skt.prune(img, min_seg_length=10, iterative_prune=3)

thislayer = Layer(name="L_000", original_img=img, dpi=300)
thislayer.islands = [Island(f"I_000", img)]

w_configurations = files.Config("welding_config.yaml")
[d_tw, frst_d_tw, sob_tw_per, name_prog_t] = gui.ask_parameters_thin_walls(w_configurations)
[void_max,
 external_max,
 internal_max,d_cont, 
 frst_d_cont, 
 sob_cont_per,
 name_prog_c] = gui.ask_parameters_offsets(w_configurations)
[n_max, 
 d_bridg, 
 frst_d_bridg, 
 sob_bridg_per, 
 name_prog_b, 
 connect_offsets] = gui.ask_parameters_bridges(w_configurations)
[d_larg, 
 frst_d_larg, 
 sob_larg_per, 
 name_prog_l, 
 large_weaving_limmit] = gui.ask_parameters_zigzags(w_configurations)
[vel_vazio, 
 p_entre_int_ext, 
 p_entre_layers,
 substratoy, 
 substratox, 
 cortey, 
 cortex] = gui.ask_parameters_Gcode()
coords_substrato = [substratoy, substratox]
coords_corte = [cortey, cortex]

thislayer.base_frame = img.shape
thislayer.pxl_per_mm = thislayer.dpi / 25.4
thislayer.mm_per_pxl = 1 / thislayer.pxl_per_mm
thislayer.islands[0].contours = Offset("O_000", np.zeros_like(thislayer.original_img), [])
thislayer.islands[0].contours.regions = []
thislayer.program_cont = name_prog_c
thislayer.program_bridg = name_prog_b
thislayer.program_tw = name_prog_t
thislayer.program_larg = name_prog_l
thislayer.void_max = "teste"
thislayer.max_internal_walls = "teste"
thislayer.max_external_walls = "teste"
thislayer.n_max = n_max
thislayer.n_layers = 1
layers = [thislayer]

sequence = path_tools.img_to_chain(thislayer.original_img)
if len(sequence) > 0 and isinstance(sequence[0][0], (list, tuple, np.ndarray)):
    new_seq = []
    for line in sequence:
        # determine a suitable first point for the line using hit-miss ends
        line_img = it.points_to_img(line, np.zeros_like(thislayer.original_img))
        ends = pt.img_to_points(mt.hitmiss_ends_v2(line_img))
        if len(ends) > 0:
            first_pt = ends[0]
            line = path_tools.set_first_pt_in_seq(line, first_pt)
        simplified = path_tools.simplifica_retas_masterV2(
            path_tools.cut_repetition(line), 1, [])
        new_seq.append(simplified)
    
    colored_new_seg = np.zeros_like(thislayer.original_img)
    color = 1
    for route in new_seq:
        colored_new_seg = it.chain_to_lines(route, colored_new_seg.astype(np.uint8),color=color) 
        color += 1
    a = 8
    b = 9
    new_seq[a],new_seq[b] = new_seq[b],new_seq[a]

    new_seq[9] = new_seq[9][::-1]


    sequence = [pt for line in new_seq for pt in line]
aaaa = it.points_to_img(sequence, np.zeros_like(thislayer.original_img))
layers[0].islands[0].contours.pts_cont = sequence
layers[0].islands[0].island_route = sequence
layers[0].islands[0].thinwalls_tree_route = []
layers[0].islands[0].internal_tree_route = []
layers[0].islands[0].external_tree_route = []


with Timer(f"Step 10R"):
    gcode_tools.layers_to_Gcode_UFSC(layers, 
                                    folders, 
                                    w_configurations,
                                    vel_vazio,
                                    p_entre_int_ext,
                                    p_entre_layers,
                                    coords_substrato,
                                    coords_corte,
                                    flag_drawing=True
                                    )
%reset_selective -f "layers"